# Chapter 9: Retrieval — Putting It to Work
## Topic 2: Query Transformation for Hinglish

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every retrieval topic so far has referenced the same finding as background context: this project's email corpus is majority Hinglish, and the knowledge base's policy documents are formal English. This topic is where that finding is finally addressed head-on as its own dedicated engineering problem, rather than a side note.
- Query transformation, in general, means rewriting a query before it reaches retrieval to make it a better retrieval input. This chapter's earlier Chapter 8 work covered HyDE as one general-purpose query rewriting technique. This topic is narrower and more targeted: transformation specifically aimed at bridging Hinglish vocabulary to the English vocabulary the knowledge base actually uses.
- Why a dedicated, targeted technique is worth having alongside a general one like HyDE: this project's vocabulary mismatch problem is not a vague, hard-to-characterize issue — it's a specific, enumerable, already-measured set of term pairs (byaj → interest, machurity → maturity, nikalna → withdrawal). A general-purpose rewriting technique pays a real per-query model-call cost to solve a problem that, for this specific and already-understood term list, can be solved almost for free with a lookup table.


### 2. Internal Working — Step by Step

**The core mechanism: term-level substitution/expansion before retrieval**

1. Maintain an explicit Hinglish-to-English term mapping — built from real, measured term frequency data in the actual email corpus (not guessed at), exactly mirroring how the existing `fd_keyword_groups.txt` file was itself derived from real term-frequency counts against FD-labeled rows.
2. Tokenize the incoming query and check each token against the mapping.
3. For every matched Hinglish term, append (not replace) its English equivalent to the query — expansion, not substitution, preserves the original term in case it happens to have some residual signal value, while adding the term that the knowledge base's vocabulary actually uses.
4. Pass the expanded query to retrieval exactly as before — this transformation happens entirely upstream of Chapter 7's retrieval pipeline, which is completely unchanged; only the query text it receives is different.

**Where this fits with BM25 specifically (the connection back to Chapter 7 Topic 1):**

- BM25 scores a query at exactly zero against a document when there is zero token overlap. A raw Hinglish query like "FD ka byaj kab milega" has zero overlap with a document saying "the interest rate is credited quarterly" — no shared tokens at all.
- After term expansion, the query becomes something like "FD ka byaj interest kab milega maturity" — now "interest" and "maturity" are present as real tokens, giving BM25 actual overlapping vocabulary to score against, where before it had structurally nothing to work with.
- This is a direct, mechanical fix for the specific zero-score failure mode Chapter 7 Topic 1 demonstrated with real numbers.

**Where this fits with dense retrieval:**

- Dense retrieval already partially bridges vocabulary gaps through learned semantic similarity, but Chapter 7 Topic 2 measured this project's own discrimination gap at only about 0.039 between genuinely related and unrelated content for short, ambiguous Hinglish text — a dangerously thin margin. Term expansion adds an additional, more literal signal on top of whatever semantic bridging the embedding model already provides, rather than relying on embeddings alone to close a gap they only partially close.


### 3. How This Is Implemented in This Project

- The term mapping is built directly from the existing `fd_keyword_groups.txt` and `fd_negation_phrases.txt` files — these already encode exactly the kind of Hinglish-to-English domain term relationships this topic needs, just structured for the rule-based classifier's use case rather than retrieval's. Reformatting this same, already-validated data as an explicit bidirectional term map is a natural, low-cost reuse of existing project assets rather than building a new resource from scratch.
- The transformation step runs immediately before retrieval is triggered (Topic 1) — once a query has been determined to warrant retrieval, term expansion runs on it as the very first retrieval-specific processing step, before BM25 tokenization and before the query is embedded for dense retrieval.
- Both the keyword-based and dense retrieval paths in the existing hybrid pipeline receive the *same* expanded query — consistent with the Chapter 7 Topic 8 principle that any transformation applied before fusion must be applied identically across every retriever being fused, not selectively to just one.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Coverage is inherently incomplete:** a term mapping built from currently-observed vocabulary will always lag behind genuinely new slang, spelling variants, or code-mixing patterns that emerge after the mapping was built. This is a maintenance discipline, not a one-time task — the mapping needs periodic review against fresh production query samples, mirroring how the original `fd_keyword_groups.txt` file itself needed real term-frequency analysis to build in the first place.
- **Spelling variant explosion:** Hinglish transliteration has no single standard spelling — "machurity," "maturity," "muchurity" might all appear for the same underlying word. A term mapping keyed on exact string match misses variants it wasn't explicitly built to catch; a more robust implementation needs either a broader variant list per canonical term or a fuzzy-matching layer on top of exact lookup.
- **False-positive expansion:** a Hinglish term that coincidentally matches an unrelated English word fragment risks adding a spurious, irrelevant expansion term to the query — this needs to be checked against real query samples during mapping construction, not assumed safe by default.
- **Latency and cost:** this is close to the cheapest possible query transformation technique available — a dictionary lookup over a tokenized query, with no model call at all. This is a meaningful advantage over HyDE specifically for this well-characterized problem, since it adds essentially zero latency and zero marginal cost per query.
- **Monitoring:** track what fraction of triggered retrieval queries actually match at least one term in the mapping — a low or declining match rate over time is an early signal that real query vocabulary has drifted away from what the mapping currently covers, and the mapping needs refreshing.
- **Interaction with query contextualization (Chapter 8 Topic 6) and HyDE (Chapter 8 Topic 7):** these techniques are not mutually exclusive and compose naturally — a multi-turn follow-up can first be contextualized using conversation history, then have its resulting self-contained query run through Hinglish term expansion, and if the corpus's vocabulary mismatch problem turns out to need more than simple term substitution can provide, HyDE can run on the already-expanded query as a further, more general-purpose enhancement.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **Term expansion vs full HyDE for this specific problem:** term expansion is far cheaper and directly targets a known, already-measured, enumerable vocabulary gap. HyDE is more general-purpose and can help with a wider range of phrasing problems, but at meaningfully higher per-query cost. Given that vocabulary mismatch is this project's specifically diagnosed, dominant retrieval weakness, term expansion likely captures most of the achievable benefit for a fraction of HyDE's cost — this should be validated head-to-head on the evaluation harness (Chapter 7 Topic 9), not assumed.
- **Substitution vs expansion (append):** replacing the original Hinglish term entirely with its English equivalent risks losing any residual signal the original term carried, and makes debugging harder (the original customer wording disappears from what retrieval actually sees). Appending the English equivalent while keeping the original preserves both signals at the cost of a slightly longer query — the safer default given the low cost of a few extra tokens.
- **How aggressively to expand:** expanding every possible matched term maximizes coverage but risks diluting the query with too many added terms if a query happens to contain several mapped words; a more conservative approach might cap the number of expansions per query, or weight expansion terms lower than original terms if the retrieval scoring supports it. This is a tunable design choice that should be validated empirically against retrieval evaluation metrics.


### 6. Alternatives and When to Use Each

- **Explicit term mapping and expansion (this topic's primary technique):** the right first choice whenever the vocabulary mismatch is well-characterized, measurable, and reasonably stable — exactly this project's situation, given the already-existing `fd_keyword_groups.txt` derived from real term-frequency data.
- **HyDE (Chapter 8 Topic 7):** worth adding on top of, or in place of, term expansion if evaluation shows the vocabulary mismatch problem is broader or more varied than a fixed term list can capture — genuinely novel phrasings, not just known term substitutions.
- **Fuzzy string matching or transliteration-normalization libraries:** worth adding specifically to address the spelling-variant-explosion edge case, if a fixed exact-match term list proves too brittle against real-world spelling variance in production traffic.
- **Multilingual embedding model upgrade (Chapter 7 Topic 2):** a different lever entirely, operating on the dense retrieval side rather than the query-text side — worth pursuing in parallel with, not instead of, query-level term expansion, since they address the same underlying problem from different layers of the pipeline.


### 7. Common Mistakes and Production Failures

- Building a term mapping from intuition or a general Hindi-English dictionary rather than from real, measured term-frequency data in the actual production query corpus — a mapping not grounded in real usage patterns will miss the terms that actually matter and may include terms that never actually appear.
- Replacing the original query term entirely instead of appending the expansion, losing potentially useful signal and making debugging harder since the original customer wording disappears from what retrieval sees.
- Applying term expansion inconsistently across the hybrid retrieval pipeline's multiple retrievers — expanding the query for keyword-based retrieval but not for dense retrieval (or vice versa) creates the same kind of inconsistent-filtering bug described in Chapter 7 Topic 8, just applied to query transformation instead of metadata filtering.
- Treating the term mapping as a one-time artifact rather than an living resource requiring periodic review against fresh query samples, letting real vocabulary drift silently reduce the mapping's coverage and effectiveness over time.
- Assuming term expansion alone fully solves the vocabulary mismatch problem without ever measuring it against the retrieval evaluation harness — some genuinely novel phrasing patterns will still need a more general technique like HyDE.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why is a targeted Hinglish term mapping worth building alongside a general-purpose technique like HyDE?
  A: The vocabulary mismatch problem in this project is specific, enumerable, and already measured from real term-frequency data — a fixed term mapping can solve most of it almost for free, via a simple dictionary lookup. HyDE is more general-purpose and handles a wider range of phrasing problems, but at meaningfully higher per-query cost, which isn't justified for a problem this well-characterized.

- Q: Why append the English equivalent rather than replacing the original Hinglish term?
  A: Appending preserves both the original term's residual signal and the new term's ability to match the knowledge base's vocabulary, at the cost of a slightly longer query. Replacing risks losing signal from the original term and makes debugging harder, since the original customer wording disappears from what retrieval actually processes.

**Intermediate**

- Q: How would you build a Hinglish-to-English term mapping for a new domain, rather than guessing at likely term pairs?
  A: Analyze real term-frequency data from the actual production query corpus, specifically comparing term frequencies between queries that are known to be about the target domain versus those that aren't — exactly how this project's existing `fd_keyword_groups.txt` file was derived. This grounds the mapping in genuine usage patterns rather than assumptions about what vocabulary customers might use.

- Q: Term expansion is applied inconsistently — the keyword-based retriever gets the expanded query, but dense retrieval still receives the raw query. What's the risk?
  A: This creates an inconsistency very similar to the metadata-filtering bug from Chapter 7 Topic 8 — the two retrievers being fused are no longer operating on the same effective input, producing an unpredictable, possibly incorrect final ranking after fusion. Any query transformation applied before retrieval must be applied identically across every retriever being combined.

**Advanced**

- Q: Design a monitoring strategy to detect when a Hinglish term mapping is becoming stale.
  A: Track the fraction of triggered retrieval queries that match at least one term in the mapping over time — a declining match rate signals that real query vocabulary is drifting away from what the mapping currently covers. Periodically sample queries that triggered retrieval but had zero mapping matches and had poor retrieval scores, and manually review them for new, unmapped vocabulary patterns worth adding to the mapping.

- Q: A teammate proposes replacing the term-mapping approach entirely with HyDE, arguing it's more general and future-proof. How do you respond?
  A: HyDE is genuinely more general-purpose, but for a vocabulary mismatch problem this specifically diagnosed and already measured, term expansion likely captures most of the achievable benefit at a small fraction of HyDE's per-query cost. The right approach is to validate both head-to-head on the retrieval evaluation harness before replacing a cheap, targeted, already-working solution with a more expensive, general one — and even if HyDE shows additional benefit, the two techniques compose naturally rather than being mutually exclusive.

**Scenario-based**

- Q: After deploying term expansion, retrieval evaluation shows improvement for known Hinglish terms in the mapping, but a new customer segment is using code-mixed phrases with regional slang not in the mapping at all. What's your plan?
  A: First quantify the scale of the problem — sample queries from this segment, measure their retrieval quality, and estimate what fraction of overall traffic they represent. If it's a small, contained segment, targeted mapping updates covering their specific vocabulary is the cheapest fix. If this segment represents a broader trend suggesting term mapping alone won't keep pace with evolving language use, this is the specific evidence needed to justify layering HyDE (or a similar general-purpose technique) on top of term expansion, rather than continuously expanding the fixed mapping indefinitely.

**Follow-up questions to expect:**

- "How would you handle a term that has multiple possible English meanings depending on context?"
- "What's the risk of over-expanding a query with too many added terms?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic is query expansion, one of the general query rewriting techniques introduced conceptually in Chapter 8 Topic 7, now applied concretely:** recognizing that this project-specific Hinglish mapping is a specific instance of the general query expansion technique (rather than a bespoke, unrelated idea) connects this topic cleanly back to the broader query rewriting taxonomy.
- **Term mapping construction is itself a small, standalone data analysis task:** deriving a genuinely useful term mapping from real term-frequency data is conceptually similar to the feature engineering and exploratory data analysis work from much earlier in the project (Chapter 1's EDA) — the same discipline of grounding decisions in measured data, rather than intuition, applies here just as much as it did there.
- **This is a narrow instance of the much broader cross-lingual and code-mixed information retrieval research area:** production systems serving genuinely multilingual or code-mixed user bases face this exact problem at much larger scale, and more sophisticated academic and industrial solutions exist (transliteration normalization, cross-lingual embedding alignment) beyond what a fixed term mapping can offer — worth knowing this broader research area exists even if a simpler technique suffices for the current problem's scale.

### 10. Quick Revision Summary (for last-minute interview prep)

> Query transformation for Hinglish is a targeted, cheap alternative to general-purpose query rewriting (like HyDE), built specifically around this project's already-measured, enumerable vocabulary mismatch problem. The mechanism: maintain an explicit Hinglish-to-English term mapping derived from real term-frequency data (reusing the same discipline behind the existing `fd_keyword_groups.txt` file), and append matched terms' English equivalents to the query before it reaches retrieval — this directly fixes BM25's zero-score failure mode for vocabulary-mismatched queries, and adds a literal signal on top of whatever partial semantic bridging dense retrieval's embeddings already provide. This transformation must be applied consistently across every retriever in the hybrid pipeline, exactly like any pre-fusion transformation. It's nearly free compared to HyDE, but requires ongoing maintenance as real vocabulary drifts, and doesn't fully replace more general techniques for genuinely novel phrasing patterns beyond the mapped term list — the two approaches compose rather than compete.


### Module 1: Setup — Term Mapping Derived from the Project's Real `fd_keyword_groups.txt`

Build the Hinglish-to-English term mapping from the project's actual keyword groups file, not a guessed dictionary.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- term mapping from real project data
#
# These term pairs are derived from the ACTUAL fd_keyword_groups.txt
# file in this project's repository, which was itself built from real
# term-frequency counts against FD-labeled rows in fd_dataset_messy.csv.
# We are not guessing at plausible Hinglish terms -- these are the
# project's own measured, real vocabulary.
# ------------------------------------------------------------------

import numpy as np
from rank_bm25 import BM25Okapi

# Hinglish term -> English equivalent, built from the real project file
HINGLISH_TO_ENGLISH = {
    "machurity": "maturity",
    "byaj": "interest",
    "nikalna": "withdrawal",
    "naya": "new",          # from "naya fd" -> new FD
    "jama": "deposited",    # from "jama kiya" -> deposited
    "kiya": "deposited",
}

def tokenize(text: str) -> list:
    return text.lower().split()


def expand_query(query: str, term_map: dict) -> str:
    """Appends the English equivalent for every matched Hinglish term,
    keeping the original terms too (expansion, not substitution)."""
    tokens = tokenize(query)
    expanded_tokens = list(tokens)
    for token in tokens:
        if token in term_map and term_map[token] not in expanded_tokens:
            expanded_tokens.append(term_map[token])
    return " ".join(expanded_tokens)


TEST_QUERIES = [
    "FD ka byaj kab milega",                     # "when will my FD interest arrive"
    "machurity ka paisa nahi aaya mera account",  # "maturity money hasn't come to my account"
    "naya fd nikalna hai jaldi",                  # "want a new FD, withdraw quickly"
]

print("=" * 70)
print("MODULE 1: HINGLISH TERM MAPPING (from real project data)")
print("=" * 70)
print("Mapping (Hinglish -> English):")
for hi, en in HINGLISH_TO_ENGLISH.items():
    print(f"  {hi:<12} -> {en}")

print("\nQuery expansion examples:")
for query in TEST_QUERIES:
    expanded = expand_query(query, HINGLISH_TO_ENGLISH)
    print(f"\n  Original: '{query}'")
    print(f"  Expanded: '{expanded}'")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: HINGLISH TERM MAPPING (from real project data)
Mapping (Hinglish -> English):
  machurity    -> maturity
  byaj         -> interest
  nikalna      -> withdrawal
  naya         -> new
  jama         -> deposited
  kiya         -> deposited

Query expansion examples:

  Original: 'FD ka byaj kab milega'
  Expanded: 'fd ka byaj kab milega interest'

  Original: 'machurity ka paisa nahi aaya mera account'
  Expanded: 'machurity ka paisa nahi aaya mera account maturity'

  Original: 'naya fd nikalna hai jaldi'
  Expanded: 'naya fd nikalna hai jaldi new withdrawal'

Module 1 complete. Run Module 2 next.


### Module 2: BM25 Retrieval — Before vs After Expansion

Run real BM25 retrieval on the raw Hinglish queries and the expanded queries against a small English knowledge base, and measure the concrete difference.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: BM25 retrieval quality -- raw vs expanded queries
# ------------------------------------------------------------------

KNOWLEDGE_BASE = [
    "The maturity payout including principal and interest is credited quarterly to the registered bank account.",
    "Premature withdrawal of a Fixed Deposit incurs a 1 percent penalty on the applicable interest rate.",
    "A new FD can be opened at any branch with a minimum deposit of ten thousand rupees.",
    "Nomination facility allows a nominee to be registered for the tenure of the Fixed Deposit.",
]

tokenized_corpus = [tokenize(doc) for doc in KNOWLEDGE_BASE]
bm25 = BM25Okapi(tokenized_corpus)

print("=" * 70)
print("BM25 RETRIEVAL: RAW HINGLISH QUERY vs EXPANDED QUERY")
print("=" * 70)

zero_score_count_raw = 0
zero_score_count_expanded = 0

for query in TEST_QUERIES:
    expanded = expand_query(query, HINGLISH_TO_ENGLISH)

    raw_scores = bm25.get_scores(tokenize(query))
    expanded_scores = bm25.get_scores(tokenize(expanded))

    raw_best = max(raw_scores)
    expanded_best = max(expanded_scores)

    if raw_best == 0.0:
        zero_score_count_raw += 1
    if expanded_best == 0.0:
        zero_score_count_expanded += 1

    print(f"\nQuery: '{query}'")
    print(f"  Raw query best BM25 score:      {raw_best:.4f}")
    print(f"  Expanded query best BM25 score: {expanded_best:.4f}")
    if raw_best == 0.0 and expanded_best > 0.0:
        best_doc_idx = int(np.argmax(expanded_scores))
        print(f"  -> FIXED a zero-score case! Expanded query now matches:")
        print(f"     '{KNOWLEDGE_BASE[best_doc_idx][:65]}...'")

print(f"\n{zero_score_count_raw} of {len(TEST_QUERIES)} raw queries scored ZERO against every document.")
print(f"{zero_score_count_expanded} of {len(TEST_QUERIES)} expanded queries scored ZERO against every document.")

if zero_score_count_expanded < zero_score_count_raw:
    print("\nCONFIRMED: term expansion directly fixed BM25's zero-score failure")
    print("mode for at least one genuinely vocabulary-mismatched query -- this")
    print("is the exact mechanism described in the theory, demonstrated with")
    print("real numbers from real project keyword data.")

print("\nModule 2 complete. Run Module 3 next.")


BM25 RETRIEVAL: RAW HINGLISH QUERY vs EXPANDED QUERY

Query: 'FD ka byaj kab milega'
  Raw query best BM25 score:      0.8181
  Expanded query best BM25 score: 0.8181

Query: 'machurity ka paisa nahi aaya mera account'
  Raw query best BM25 score:      0.0000
  Expanded query best BM25 score: 0.8659
  -> FIXED a zero-score case! Expanded query now matches:
     'The maturity payout including principal and interest is credited ...'

Query: 'naya fd nikalna hai jaldi'
  Raw query best BM25 score:      0.8181
  Expanded query best BM25 score: 1.6362

1 of 3 raw queries scored ZERO against every document.
0 of 3 expanded queries scored ZERO against every document.

CONFIRMED: term expansion directly fixed BM25's zero-score failure
mode for at least one genuinely vocabulary-mismatched query -- this
is the exact mechanism described in the theory, demonstrated with
real numbers from real project keyword data.

Module 2 complete. Run Module 3 next.


### Module 3: Consistency Check — Expansion Applied to Both Retrievers in the Hybrid Pipeline

Demonstrates the theory's warning concretely: applying expansion to only ONE retriever in a hybrid setup produces an inconsistent, unpredictable fused ranking.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Consistency across the hybrid pipeline (BM25 + dense)
#
# Reuses the RRF fusion logic from Chapter 7 Topic 4 to show the same
# class of bug as Chapter 7 Topic 8's metadata-filtering inconsistency,
# just applied to query TRANSFORMATION instead of filtering.
# ------------------------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def normalize_vector(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def cosine_similarity(a, b):
    d = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / d) if d > 0 else 0.0

def reciprocal_rank_fusion(ranked_lists: list, k: int = 60) -> dict:
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for position, doc_id in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + position)
    return rrf_scores


test_query = "machurity ka paisa nahi aaya mera account"
expanded_query = expand_query(test_query, HINGLISH_TO_ENGLISH)

# build dense embedding space over knowledge base + both query variants
all_texts = KNOWLEDGE_BASE + [test_query, expanded_query]
vectorizer = TfidfVectorizer()
sparse = vectorizer.fit_transform(all_texts)
svd = TruncatedSVD(n_components=4, random_state=42)
dense = svd.fit_transform(sparse)
dense_norm = np.array([normalize_vector(v) for v in dense])

kb_vectors = dense_norm[:len(KNOWLEDGE_BASE)]
raw_query_vec = dense_norm[len(KNOWLEDGE_BASE)]
expanded_query_vec = dense_norm[len(KNOWLEDGE_BASE) + 1]

# BM25 rankings (both variants)
bm25_raw_scores = bm25.get_scores(tokenize(test_query))
bm25_expanded_scores = bm25.get_scores(tokenize(expanded_query))
bm25_raw_ranking = list(np.argsort(bm25_raw_scores)[::-1])
bm25_expanded_ranking = list(np.argsort(bm25_expanded_scores)[::-1])

# Dense rankings (both variants)
dense_raw_scores = [cosine_similarity(raw_query_vec, v) for v in kb_vectors]
dense_expanded_scores = [cosine_similarity(expanded_query_vec, v) for v in kb_vectors]
dense_raw_ranking = list(np.argsort(dense_raw_scores)[::-1])
dense_expanded_ranking = list(np.argsort(dense_expanded_scores)[::-1])

print("=" * 70)
print("CONSISTENT vs INCONSISTENT QUERY EXPANSION IN HYBRID RETRIEVAL")
print("=" * 70)
print(f"Query: '{test_query}'")
print(f"Expanded: '{expanded_query}'\n")

# INCONSISTENT: BM25 gets expanded query, dense retrieval still gets RAW query
inconsistent_fused = reciprocal_rank_fusion([bm25_expanded_ranking, dense_raw_ranking])
inconsistent_top = sorted(inconsistent_fused.keys(), key=lambda d: inconsistent_fused[d], reverse=True)

# CONSISTENT: BOTH retrievers get the expanded query
consistent_fused = reciprocal_rank_fusion([bm25_expanded_ranking, dense_expanded_ranking])
consistent_top = sorted(consistent_fused.keys(), key=lambda d: consistent_fused[d], reverse=True)

print("INCONSISTENT (BM25 expanded, dense NOT expanded) -- fused ranking:")
for idx in inconsistent_top:
    print(f"  Doc {idx} | {KNOWLEDGE_BASE[idx][:60]}...")

print("\nCONSISTENT (BOTH retrievers expanded) -- fused ranking:")
for idx in consistent_top:
    print(f"  Doc {idx} | {KNOWLEDGE_BASE[idx][:60]}...")

if inconsistent_top != consistent_top:
    print("\nCONFIRMED: the fused ranking genuinely DIFFERS depending on whether")
    print("expansion was applied consistently across both retrievers -- this is")
    print("exactly the same class of bug as Chapter 7 Topic 8's inconsistent")
    print("metadata filtering, now shown for query transformation specifically.")
else:
    print("\nIn this specific run, the top ranking happened to match either way --")
    print("the risk of inconsistency is still real and depends on the specific")
    print("query and corpus; always apply transformations identically regardless.")

print("\nModule 3 complete. All Chapter 9 Topic 2 modules done.")
print()
print("=" * 70)
print("CHAPTER 9 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Term mapping built from REAL term-frequency data (like this project's
  own fd_keyword_groups.txt), not guessed vocabulary.

  Expansion (append), not substitution -- keep the original term AND
  add its English equivalent.

  This directly fixes BM25's zero-score failure mode for vocabulary-
  mismatched queries -- a mechanical, measurable effect.

  MUST be applied consistently across every retriever in a hybrid
  pipeline before fusion -- inconsistent application is the same class
  of bug as inconsistent metadata filtering (Chapter 7 Topic 8).

  Nearly free compared to HyDE -- the right first fix for a
  well-characterized, already-measured vocabulary mismatch problem.
""")


CONSISTENT vs INCONSISTENT QUERY EXPANSION IN HYBRID RETRIEVAL
Query: 'machurity ka paisa nahi aaya mera account'
Expanded: 'machurity ka paisa nahi aaya mera account maturity'

INCONSISTENT (BM25 expanded, dense NOT expanded) -- fused ranking:
  Doc 0 | The maturity payout including principal and interest is cred...
  Doc 3 | Nomination facility allows a nominee to be registered for th...
  Doc 1 | Premature withdrawal of a Fixed Deposit incurs a 1 percent p...
  Doc 2 | A new FD can be opened at any branch with a minimum deposit ...

CONSISTENT (BOTH retrievers expanded) -- fused ranking:
  Doc 0 | The maturity payout including principal and interest is cred...
  Doc 2 | A new FD can be opened at any branch with a minimum deposit ...
  Doc 3 | Nomination facility allows a nominee to be registered for th...
  Doc 1 | Premature withdrawal of a Fixed Deposit incurs a 1 percent p...

CONFIRMED: the fused ranking genuinely DIFFERS depending on whether
expansion was applied consistently ac